In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join('..', '..')))

import torch
import json
import numpy as np
import pandas as pd
import itertools

from notebooks.local.utils import get_paths, create_folders, load_progress, save_progress, mark_done, is_done

PATHS    = get_paths()
DEVICE   = "cuda" if torch.cuda.is_available() else "cpu"
USE_FP16 = torch.cuda.is_available()
THRESHOLDS = [0.05, 0.10, 0.20]

K_VALUES   = [3, 5, 7, 9]
TEMP_VALUES = [0.05, 0.10, 0.20, 0.50, 1.0]

create_folders(PATHS)
print("Device:", DEVICE)


In [ ]:
import os
import shutil, tarfile
from notebooks.local.utils import get_paths, create_folders, download_file

PATHS    = get_paths()
create_folders(PATHS)

spair_check = os.path.join(PATHS['spair71k'], 'JPEGImages')
if not os.path.exists(spair_check):
    print('SPair-71k not found. Downloading (~2 GB) ...')
    tar_path = os.path.join(PATHS['data'], 'SPair-71k.tar.gz')
    download_file(
        'http://cvlab.postech.ac.kr/research/SPair-71k/data/SPair-71k.tar.gz',
        tar_path, desc='SPair-71k',
    )
    print('Extracting ...')
    with tarfile.open(tar_path, 'r:gz') as t:
        t.extractall(PATHS['spair71k'])
    extracted_sub = os.path.join(PATHS['spair71k'], 'SPair-71k')
    if os.path.isdir(extracted_sub):
        for item in os.listdir(extracted_sub):
            shutil.move(os.path.join(extracted_sub, item),
                        os.path.join(PATHS['spair71k'], item))
        os.rmdir(extracted_sub)
    os.remove(tar_path)
    print('SPair-71k ready.')
else:
    print('SPair-71k already present.')

if not os.path.exists(PATHS['dinov2_w']):
    print('Downloading DINOv2 ViT-B/14 weights (~330 MB) ...')
    download_file(
        'https://dl.fbaipublicfiles.com/dinov2/dinov2_vitb14/dinov2_vitb14_pretrain.pth',
        PATHS['dinov2_w'], desc='DINOv2',
    )
else:
    print('DINOv2 weights present.')

if not os.path.exists(PATHS['sam_w']):
    print('Downloading SAM ViT-B weights (~370 MB) ...')
    download_file(
        'https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth',
        PATHS['sam_w'], desc='SAM',
    )
else:
    print('SAM weights present.')

if not os.path.exists(PATHS['dinov3_w']):
    print('WARNING: DINOv3 weights not found at', PATHS['dinov3_w'])
    print('  Place dinov3_vitb16_pretrain.pth in weights/ (obtain from project maintainer).')
else:
    print('DINOv3 weights present.')


In [ ]:
from src.models.dinov2.dinov2.models.vision_transformer import vit_base as vit_base_v2
from src.models.dinov3.dinov3.models.vision_transformer import vit_base as vit_base_v3
from src.models.segment_anything.segment_anything import sam_model_registry
from src.datasets.spair_dataset import SPairDataset
from experiments.evaluate import evaluate, evaluate_multilayer, save_results


def load_finetuned_or_pretrained(backbone, paths, device, use_fp16=False):
    ft_map = {'dinov2': paths['dinov2_ft'], 'dinov3': paths['dinov3_ft'], 'sam': paths['sam_ft']}
    ft_path = ft_map[backbone]

    if backbone == 'dinov2':
        model = vit_base_v2(img_size=(518,518), patch_size=14,
                            num_register_tokens=0, block_chunks=0, init_values=1.0)
        img_size, patch_size = 518, 14
    elif backbone == 'dinov3':
        model = vit_base_v3(img_size=512, patch_size=16)
        img_size, patch_size = 512, 16
    elif backbone == 'sam':
        model = sam_model_registry['vit_b'](checkpoint=paths['sam_w'])
        img_size, patch_size = 512, 16

    if os.path.exists(ft_path) and backbone != 'sam':
        ckpt = torch.load(ft_path, map_location=device, weights_only=True)
        state = ckpt['model_state_dict'] if 'model_state_dict' in ckpt else ckpt
        model.load_state_dict(state, strict=True)
        print(f"Loaded fine-tuned {backbone}")
    else:
        if backbone != 'sam':
            ckpt_key = {'dinov2': 'dinov2_w', 'dinov3': 'dinov3_w'}[backbone]
            ckpt = torch.load(paths[ckpt_key], map_location=device, weights_only=True)
            model.load_state_dict(ckpt, strict=True)
        print(f"Using pretrained {backbone} (no fine-tune ckpt found)")

    model = model.to(device)
    if use_fp16 and backbone != 'sam':
        model = model.half()
    model.eval()
    return model, img_size, patch_size


pair_ann = os.path.join(PATHS['spair71k'], 'PairAnnotation')
layout   = os.path.join(PATHS['spair71k'], 'Layout')
images   = os.path.join(PATHS['spair71k'], 'JPEGImages')
test_dataset = SPairDataset(pair_ann, layout, images, 'large', 0.1, 'test')
print(f"Test pairs: {len(test_dataset)}")


## Grid Search — DINOv2

In [ ]:
PROGRESS_PATH = os.path.join(PATHS['step3_grid'], 'progress_dinov2.json')
progress = load_progress(PROGRESS_PATH)

model, img_size, patch_size = load_finetuned_or_pretrained('dinov2', PATHS, DEVICE, USE_FP16)

for K, temp in itertools.product(K_VALUES, TEMP_VALUES):
    key = f"K{K}_T{temp}"
    if is_done(progress, 'dinov2', 'grid', key):
        print(f"Skip dinov2 {key}")
        continue

    out_dir = os.path.join(PATHS['step3_grid'], f"dinov2_{key}")
    os.makedirs(out_dir, exist_ok=True)
    stats_path = os.path.join(out_dir, 'overall_stats.json')

    if not os.path.exists(stats_path):
        per_img, all_kp, t = evaluate(
            model, test_dataset, DEVICE, THRESHOLDS,
            use_windowed_softargmax=True, K=K, temperature=temp,
        )
        save_results(per_img, all_kp, out_dir, t, THRESHOLDS)

    mark_done(progress, 'dinov2', 'grid', key, PROGRESS_PATH)
    with open(stats_path) as f:
        s = json.load(f)
    print(f"dinov2 K={K} T={temp}: PCK@0.10={s['pck@0.10']['mean']:.2f}%")

del model; torch.cuda.empty_cache()
print("DINOv2 grid search done.")


## Grid Search — DINOv3

In [ ]:
PROGRESS_PATH = os.path.join(PATHS['step3_grid'], 'progress_dinov3.json')
progress = load_progress(PROGRESS_PATH)

model, img_size, patch_size = load_finetuned_or_pretrained('dinov3', PATHS, DEVICE, USE_FP16)

for K, temp in itertools.product(K_VALUES, TEMP_VALUES):
    key = f"K{K}_T{temp}"
    if is_done(progress, 'dinov3', 'grid', key):
        print(f"Skip dinov3 {key}")
        continue

    out_dir = os.path.join(PATHS['step3_grid'], f"dinov3_{key}")
    os.makedirs(out_dir, exist_ok=True)
    stats_path = os.path.join(out_dir, 'overall_stats.json')

    if not os.path.exists(stats_path):
        per_img, all_kp, t = evaluate(
            model, test_dataset, DEVICE, THRESHOLDS,
            use_windowed_softargmax=True, K=K, temperature=temp,
        )
        save_results(per_img, all_kp, out_dir, t, THRESHOLDS)

    mark_done(progress, 'dinov3', 'grid', key, PROGRESS_PATH)
    with open(stats_path) as f:
        s = json.load(f)
    print(f"dinov3 K={K} T={temp}: PCK@0.10={s['pck@0.10']['mean']:.2f}%")

del model; torch.cuda.empty_cache()
print("DINOv3 grid search done.")


## Best Configurations

In [ ]:
import glob

rows = []
for backbone in ['dinov2', 'dinov3']:
    best_pck = -1
    best_key = None
    best_stats = None
    pattern = os.path.join(PATHS['step3_grid'], f"{backbone}_K*", 'overall_stats.json')
    for stats_path in glob.glob(pattern):
        with open(stats_path) as f:
            s = json.load(f)
        pck = s.get('pck@0.10', {}).get('mean', -1)
        if pck > best_pck:
            best_pck = pck
            best_key = os.path.basename(os.path.dirname(stats_path))
            best_stats = s

    if best_stats:
        rows.append({
            'Model': backbone,
            'Best Config': best_key,
            'PCK@0.05': round(best_stats.get('pck@0.05',{}).get('mean', float('nan')), 2),
            'PCK@0.10': round(best_stats.get('pck@0.10',{}).get('mean', float('nan')), 2),
            'PCK@0.20': round(best_stats.get('pck@0.20',{}).get('mean', float('nan')), 2),
        })
    else:
        rows.append({'Model': backbone, 'Best Config': 'N/A', 'PCK@0.05': '-', 'PCK@0.10': '-', 'PCK@0.20': '-'})

df = pd.DataFrame(rows).set_index('Model')
print("Best soft-argmax configurations:")
print(df.to_string())
